In [ ]:
import os
import pandas as pd
from sentence_transformers import SentenceTransformer, util, InputExample, losses
import spacy
from spacy.cli import download
import re
import unicodedata
import torch
from torch.utils.data import DataLoader
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator
import math


In [ ]:
os.environ['WANDB_DISABLED'] = 'true'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Configurações do modelo
MODELS = {
    # "BERTimbau": "neuralmind/bert-base-portuguese-cased",
    "stjiris": 'stjiris/bert-large-portuguese-cased-legal-tsdae-gpl-nli-sts-MetaKD-v0' #modelo local treinado'/content/drive/MyDrive/Bruno de Oliveira/São Paulo/Comparações/PM x PE/OE x ODS/Modelo/'
    # "S-BERT": "paraphrase-multilingual-mpnet-base-v2",
    # "DistilBERT": "distiluse-base-multilingual-cased-v2"
}

In [ ]:
# Carregar os arquivos
FILE_A = "/content/drive/MyDrive/Artefatos Planejamento Urbano/Bruno de Oliveira/São Paulo/Plano de Metas (PM)/Estrtutura/PM-2-Objetivos Estratégicos.xlsx"
FILE_B = "/content/drive/MyDrive/Artefatos Planejamento Urbano/Bruno de Oliveira/São Paulo/Planejamento Estratégico (PE)/Estrutura/PE-1-ODS.xlsx"

In [ ]:
def load_data(file_a, file_b):
    df_a = pd.read_excel(file_a, header=None)
    df_b = pd.read_excel(file_b, header=None)
    return df_a.iloc[:, 0].dropna().tolist(), df_b.iloc[:, 0].dropna().tolist()

In [ ]:
def load_trainnng_data():
  # --- ETAPA 1: PREPARAÇÃO DOS DADOS ---
  # Nesta etapa, você deve carregar seus próprios dados.
  # Para este exemplo, vamos criar uma lista de dados fictícios.
  # O score de similaridade deve ser um float entre 0.0 e 1.0.
  # Se você anotou seus dados em uma escala diferente (ex: 1 a 5),
  # você precisa normalizá-los. Por exemplo, (score - 1) / 4.

  print("--- Etapa 1: Preparando os Dados ---")

  # Lista de pares de sentenças e seus scores de similaridade (formato: [sentença1, sentença2, score])
  # Este é o nosso dataset de treinamento
  dados_treino_brutos = [
      # Alta similaridade (score > 0.8)
      ["Ampliar o acesso à educação infantil em tempo integral.", "Construir 5 novas creches com período integral.", 0.9],
      ["Realizar a manutenção da malha viária da cidade.", "Executar uma operação tapa-buracos em todas as ruas.", 0.85],
      ["Digitalizar 100% dos processos administrativos da prefeitura.", "Implementar um sistema de gestão de processos eletrônicos.", 0.95],

      # Média similaridade (scores entre 0.4 e 0.6)
      ["Promover a saúde preventiva.", "Realizar campanhas de vacinação contra a gripe.", 0.6],
      ["Aumentar a arrecadação do município.", "Oferecer descontos para pagamento do IPTU em cota única.", 0.5],
      ["Incentivar o esporte amador.", "Reformar as quadras poliesportivas dos bairros.", 0.65],

      # Baixa similaridade (score < 0.2)
      ["Melhorar o transporte público.", "Construir um novo teatro municipal.", 0.05],
      ["Aumentar o efetivo da guarda municipal.", "Promover a coleta seletiva de lixo.", 0.1],
  ]

  # A biblioteca sentence-transformers espera objetos do tipo InputExample
  exemplos_de_treino = []
  for item in dados_treino_brutos:
      exemplo = InputExample(texts=[item[0], item[1]], label=float(item[2]))
      exemplos_de_treino.append(exemplo)

  print(f"Total de {len(exemplos_de_treino)} exemplos de treino criados.")

  return exemplos_de_treino


In [ ]:
def load_evaluator():
  # --- ETAPA 3: CRIAÇÃO DO AVALIADOR ---
  # É uma boa prática avaliar o modelo em dados que ele não viu durante o treino.
  # Vamos criar um pequeno conjunto de avaliação.

  print("\n--- Etapa 3: Criando o Avaliador ---")

  # Em um caso real, este seria um arquivo separado.
  dados_avaliacao_brutos = [
      ["Capacitar os servidores públicos.", "Oferecer cursos de aperfeiçoamento para o funcionalismo.", 0.9],
      ["Melhorar a iluminação pública.", "Instalar lâmpadas de LED nos postes da cidade.", 0.8],
      ["Fomentar o turismo local.", "Aumentar a frota de ônibus urbanos.", 0.15],
  ]

  sentencas1_eval = [item[0] for item in dados_avaliacao_brutos]
  sentencas2_eval = [item[1] for item in dados_avaliacao_brutos]
  scores_eval = [item[2] for item in dados_avaliacao_brutos]

  # O avaliador calculará a similaridade para os pares de avaliação e
  # comparará o resultado com os scores reais usando a correlação de Spearman.
  evaluator = EmbeddingSimilarityEvaluator(sentencas1_eval, sentencas2_eval, scores_eval)

  print("Avaliador criado com 3 exemplos de avaliação.")

  return evaluator


In [ ]:
def tunning_model(model_link):

  exemplos_de_treino = load_trainnng_data()

  # --- ETAPA 2: CONFIGURAÇÃO DO MODELO E DO TREINAMENTO ---

  print("\n--- Etapa 2: Configurando o Modelo e o Treinamento ---")

  # Carrega um modelo BERT pré-treinado para português.
  # Este será o nosso ponto de partida antes do ajuste fino.

  model = SentenceTransformer(model_link)

  # Para o treinamento de regressão, usamos o CosineSimilarityLoss.
  # Esta função de perda tentará fazer com que a similaridade de cosseno
  # entre os embeddings das sentenças se aproxime do score fornecido no label.
  train_loss = losses.CosineSimilarityLoss(model=model)

  # O DataLoader irá pegar nossos exemplos e organizá-los em lotes (batches) para o treinamento.
  train_dataloader = DataLoader(exemplos_de_treino, shuffle=True, batch_size=16)

  print(f"Modelo base '{model_link}' carregado.")
  print("Função de perda: CosineSimilarityLoss")

  evaluator = load_evaluator()


  # --- ETAPA 4: EXECUTANDO O TREINAMENTO ---

  print("\n--- Etapa 4: Executando o Fine-Tuning ---")

  # Definimos os parâmetros do treinamento
  num_epochs = 4
  warmup_steps = math.ceil(len(train_dataloader) * num_epochs * 0.1) # 10% de warmup
  output_path = '/content/drive/MyDrive/Artefatos Planejamento Urbano/Bruno de Oliveira/São Paulo/Comparações/PM x PE/OE x ODS/Modelo/'

  # Iniciamos o treinamento!
  model.fit(train_objectives=[(train_dataloader, train_loss)],
            evaluator=evaluator,
            epochs=num_epochs,
            warmup_steps=warmup_steps,
            output_path=output_path,
            evaluation_steps=5, # Avalia a cada 5 passos de treino
            show_progress_bar=True)


  # --- ETAPA 5: CARREGANDO E USANDO O MODELO AJUSTADO ---

  return output_path

In [ ]:
def compute_similarities_with_fine_tunning(sentences_a_clean, sentences_b_clean, model_name, sentences_a, sentences_b):
    path_modelo_ajustado = tunning_model(MODELS[model_name])
    print(f"Processando com o modelo: {model_name}")
    model = SentenceTransformer(path_modelo_ajustado)
    embeddings_a = model.encode(sentences_a_clean, convert_to_tensor=True, normalize_embeddings=True, batch_size=16)
    embeddings_b = model.encode(sentences_b_clean, convert_to_tensor=True, normalize_embeddings=True, batch_size=16)

    results_a_to_b = []
    for i, emb_a in enumerate(embeddings_a):
        similarities = util.pytorch_cos_sim(emb_a, embeddings_b).flatten()
        sorted_indices = similarities.argsort(descending=True)
        results_a_to_b.append([
            (j + 1, sentences_b[j], similarities[j].item()) for j in sorted_indices
        ])

    results_b_to_a = []
    for i, emb_b in enumerate(embeddings_b):
        similarities = util.pytorch_cos_sim(emb_b, embeddings_a).flatten()
        sorted_indices = similarities.argsort(descending=True)
        results_b_to_a.append([
            (j + 1, sentences_a[j], similarities[j].item()) for j in sorted_indices
        ])

    return results_a_to_b, results_b_to_a

In [ ]:
# def compute_similarities(sentences_a_clean, sentences_b_clean, model_name, sentences_a, sentences_b):
#     print(f"Processando com o modelo: {model_name}")
#     model = SentenceTransformer(MODELS[model_name])
#     embeddings_a = model.encode(sentences_a_clean, convert_to_tensor=True, normalize_embeddings=True, batch_size=16)
#     embeddings_b = model.encode(sentences_b_clean, convert_to_tensor=True, normalize_embeddings=True, batch_size=16)

#     results_a_to_b = []
#     for i, emb_a in enumerate(embeddings_a):
#         similarities = util.pytorch_cos_sim(emb_a, embeddings_b).flatten()
#         sorted_indices = similarities.argsort(descending=True)
#         results_a_to_b.append([
#             (j + 1, sentences_b[j], similarities[j].item()) for j in sorted_indices
#         ])

#     results_b_to_a = []
#     for i, emb_b in enumerate(embeddings_b):
#         similarities = util.pytorch_cos_sim(emb_b, embeddings_a).flatten()
#         sorted_indices = similarities.argsort(descending=True)
#         results_b_to_a.append([
#             (j + 1, sentences_a[j], similarities[j].item()) for j in sorted_indices
#         ])

#     return results_a_to_b, results_b_to_a

In [ ]:
def save_results(results_a_to_b, results_b_to_a, model_name, sentences_a, sentences_b):
    # Salvar resultados A → B
    data_a_to_b = []
    for i, row in enumerate(results_a_to_b):
        best_match = None
        # Find the best match with similarity >= 0.42
        for idx, sent, score in row:
            if score >= 0.42:
                best_match = (idx, sent, score)
                break # Stop at the first best match (highest score)

        if best_match:
            # Include only the best match if it meets the threshold
            a_row = [sentences_a[i], f"{best_match[0]}: {best_match[1]} ({best_match[2]:.2f})"]
        else:
            # If no match meets the threshold, include only the original sentence
            a_row = [sentences_a[i], ""]

        data_a_to_b.append(a_row)

    df_a_to_b = pd.DataFrame(data_a_to_b, columns=["Frase A", "Melhor Similaridade (>= 0.42)"])
    df_a_to_b.to_excel(f"Resultados_A_to_B_{model_name}_filtrado_v3.xlsx", index=False)

    # Salvar resultados B → A
    data_b_to_a = []
    for i, row in enumerate(results_b_to_a):
        best_match = None
        # Find the best match with similarity >= 0.42
        for idx, sent, score in row:
            if score >= 0.42:
                best_match = (idx, sent, score)
                break # Stop at the first best match (highest score)

        if best_match:
            # Include only the best match if it meets the threshold
            b_row = [sentences_b[i], f"{best_match[0]}: {best_match[1]} ({best_match[2]:.2f})"]
        else:
            # If no match meets the threshold, include only the original sentence
            b_row = [sentences_b[i], ""]

        data_b_to_a.append(b_row)

    df_b_to_a = pd.DataFrame(data_b_to_a, columns=["Frase B", "Melhor Similaridade (>= 0.42)"])
    df_b_to_a.to_excel(f"Resultados_B_to_A_{model_name}_filtrado_v3.xlsx", index=False)

In [ ]:
# def clean_text(text):
#     # Lowercase
#     text = text.lower()

#     # Remove acentos (opcional)
#     text = unicodedata.normalize("NFKD", text).encode("ASCII", "ignore").decode("utf-8")

#     # Remove pontuações
#     text = re.sub(r"[^\w\s]", " ", text)

#     # Remove números (se for irrelevante)
#     text = re.sub(r"\d+", "", text)

#     # Espaços duplicados
#     text = re.sub(r"\s+", " ", text).strip()

#     return text

In [ ]:
def preprocess(sentences, nlp=None):
    # cleaned = []
    # for sent in sentences:
    #     sent = clean_text(sent)
    #     doc = nlp(sent)

    #     # Lematização + remoção leve de stopwords
    #     lemmas = [token.lemma_ for token in doc if not token.is_punct and not token.is_space and not token.is_stop]

    #     # Se estiver muito curto, mantemos a versão limpa original
    #     final = " ".join(lemmas) if len(lemmas) >= 3 else sent
    #     cleaned.append(final)
    # return cleaned

    # return [' '.join([token.lemma_ for token in nlp(sent.lower()) if not token.is_stop and not token.is_punct]) for sent in sentences]

    return sentences

In [ ]:
def main():
    sentences_a, sentences_b = load_data(FILE_A, FILE_B)
    # download("pt_core_news_md")
    # nlp = spacy.load("pt_core_news_md")

    sentences_a_clean = preprocess(sentences_a)#, nlp)
    sentences_b_clean = preprocess(sentences_b)#, nlp)

    for model_name in MODELS:  # Itera sobre BERTimbau, S-BERT e DistilBERT
        results_a_to_b, results_b_to_a = compute_similarities_with_fine_tunning(sentences_a_clean, sentences_b_clean, model_name, sentences_a, sentences_b)
        save_results(results_a_to_b, results_b_to_a, model_name, sentences_a, sentences_b)
        print(f"Resultados salvos para o modelo {model_name}.")

In [ ]:
main()
